### Currency DataType

In [34]:
import polars as pl
from decimal import Decimal, ROUND_HALF_UP
from typing import Union, Optional,Final
import json

class Currency:
    """Currency class for handling monetary values with decimal precision."""
    
    SUPPORTED_CURRENCIES:Final[dict[str,str]] = {
        "USD": "$",
        "EUR": "€", 
        "SCR": "SCR"
    }
    
    def __init__(self, amount: Union[float, int, str, Decimal], currency_code: str = "USD"):
        # Validate currency code first
        if not isinstance(currency_code, str):
            raise ValueError("currency_code must be a string")
        if not currency_code.isupper():
            raise ValueError("currency_code must be uppercase")
        if len(currency_code) != 3:
            raise ValueError("currency_code must be exactly 3 characters long")
        if currency_code not in self.SUPPORTED_CURRENCIES:
            raise ValueError(f"currency_code must be one of the supported currencies: {list(self.SUPPORTED_CURRENCIES.keys())}")
        
        self.currency_code = currency_code
        
        # Handle amount conversion to Decimal for precision
        if not isinstance(amount, (float, int, str, Decimal)):
            raise ValueError("amount must be a float, int, string, or Decimal")
        
        # Convert to Decimal for precision, handling floats carefully
        if isinstance(amount, float):
            # Convert float to string to avoid floating point precision issues
            self.amount = Decimal(str(amount)).quantize(Decimal('0.0001'), rounding=ROUND_HALF_UP)
        else:
            self.amount = Decimal(str(amount)).quantize(Decimal('0.0001'), rounding=ROUND_HALF_UP)
    
    @property
    def symbol(self) -> str:
        return self.SUPPORTED_CURRENCIES[self.currency_code]
    
    def __str__(self) -> str:
        return f"{self.symbol}{self.amount}"
    
    def __repr__(self) -> str:
        return f"Currency({self.amount}, '{self.currency_code}')"
    
    def __add__(self, other: 'Currency') -> 'Currency':
        if not isinstance(other, Currency):
            raise TypeError("Can only add Currency to Currency")
        if self.currency_code != other.currency_code:
            raise ValueError("Cannot add different currencies")
        return Currency(self.amount + other.amount, self.currency_code)
    
    def __sub__(self, other: 'Currency') -> 'Currency':
        if not isinstance(other, Currency):
            raise TypeError("Can only subtract Currency from Currency")
        if self.currency_code != other.currency_code:
            raise ValueError("Cannot subtract different currencies")
        return Currency(self.amount - other.amount, self.currency_code)
    
    def __mul__(self, other: Union[int, float, Decimal]) -> 'Currency':
        if not isinstance(other, (int, float, Decimal)):
            raise TypeError("Can only multiply Currency by numeric types")
        return Currency(self.amount * Decimal(str(other)), self.currency_code)
    
    def __truediv__(self, other: Union[int, float, Decimal]) -> 'Currency':
        if not isinstance(other, (int, float, Decimal)):
            raise TypeError("Can only divide Currency by numeric types")
        if other == 0:
            raise ZeroDivisionError("Cannot divide by zero")
        return Currency(self.amount / Decimal(str(other)), self.currency_code)
    
    def __eq__(self, other) -> bool:
        return (isinstance(other, Currency) and 
                self.currency_code == other.currency_code and 
                self.amount == other.amount)
    
    def __lt__(self, other: 'Currency') -> bool:
        if not isinstance(other, Currency):
            raise TypeError("Can only compare Currency to Currency")
        if self.currency_code != other.currency_code:
            raise ValueError("Cannot compare different currencies")
        return self.amount < other.amount
    
    def __le__(self, other: 'Currency') -> bool:
        return self == other or self < other
    
    def __gt__(self, other: 'Currency') -> bool:
        return not self <= other
    
    def __ge__(self, other: 'Currency') -> bool:
        return not self < other
    
    def to_dict(self) -> dict:
        """Convert Currency to dictionary for Polars struct compatibility."""
        return {
            "amount": float(self.amount),
            "code": self.currency_code,
            "symbol": self.symbol
        }
    
    @classmethod
    def from_dict(cls, data: dict) -> 'Currency':
        """Create Currency from dictionary."""
        return cls(data["amount"], data["code"])


# Custom Polars extension for Currency handling
class CurrencyExtension:
    """Extension methods for working with Currency in Polars DataFrames."""
    
    @staticmethod
    def create_currency_schema() -> pl.Struct:
        """Create the Polars struct schema for currency data."""
        return pl.Struct([
            pl.Field("amount", pl.Float64),
            pl.Field("code", pl.Utf8),
            pl.Field("symbol", pl.Utf8)
        ])
    
    @staticmethod
    def from_currency_objects(currencies: list[Currency]) -> pl.DataFrame:
        """Convert list of Currency objects to Polars DataFrame."""
        data = [curr.to_dict() for curr in currencies]
        schema = CurrencyExtension.create_currency_schema()
        return pl.DataFrame(data, schema={"currency": schema})
    
    @staticmethod
    def to_currency_objects(df: pl.DataFrame, column: str) -> list[Currency]:
        """Convert Polars DataFrame column to Currency objects."""
        return [Currency.from_dict(row) for row in df[column].to_list()]


# Example usage and utilities
def create_currency_dataframe(data: list[dict]) -> pl.DataFrame:
    """
    Create a Polars DataFrame with currency data.
    
    Args:
        data: List of dictionaries with 'amount' and 'code' keys
    
    Returns:
        Polars DataFrame with currency struct column
    """
    currencies = [Currency(item["amount"], item.get("code", "USD")) for item in data]
    currency_dicts = [curr.to_dict() for curr in currencies]
    
    schema = CurrencyExtension.create_currency_schema()
    return pl.DataFrame({"price": currency_dicts}, schema={"price": schema})


def perform_currency_operations(df: pl.DataFrame, column: str = "price") -> pl.DataFrame:
    """
    Perform operations on currency data in a Polars DataFrame.
    
    Args:
        df: DataFrame with currency struct column
        column: Name of the currency column
    
    Returns:
        DataFrame with additional computed columns
    """
    return df.with_columns([
        # Extract individual fields
        pl.col(column).struct.field("amount").alias("amount"),
        pl.col(column).struct.field("code").alias("currency_code"),
        pl.col(column).struct.field("symbol").alias("symbol"),
        
        # Create formatted string representation
        (pl.col(column).struct.field("symbol") + 
         pl.col(column).struct.field("amount").cast(pl.Utf8)).alias("formatted_price"),
        
        # Convert to cents (for USD/EUR) for integer arithmetic
        (pl.col(column).struct.field("amount") * 100).cast(pl.Int64).alias("amount_cents")
    ])


class CurrencyConverter:
    """Utitlity for converting dataframe columns to currency datatype"""

    @staticmethod
    def cast_to_currency_struct(
        df: pl.DataFrame,
        column: str,
        currency_code: str = "USD",
        new_column_name: Optional[str] = None
    ) -> pl.DataFrame:
        """        Cast a DataFrame column to a Currency struct type.
        Args:
            df: Input Polars DataFrame
            column: Column name to convert
            currency_code: Currency code to use for conversion
            new_column_name: Optional new column name for the converted data
        Returns:
            Polars DataFrame with the specified column converted to Currency struct
        """
        if new_column_name is None:
            new_column_name = column

        # Get currency symbol

        currency_symbol = Currency.SUPPORTED_CURRENCIES.get(currency_code)
        
        
        # Convert the specified column to Currency struct
        return df.with_columns(
            pl.struct([
                pl.col(column).cast(pl.Float64).alias("amount"),
                pl.lit(currency_code).alias("code"),
                pl.lit(currency_symbol).alias("symbol")
            ]).alias(new_column_name)
        
        )
    
    @staticmethod
    def cast_multiple_columns_to_currency(
        df: pl.DataFrame,
        columns: list[str],
        currency_code: str = "USD"
    ) -> pl.DataFrame:
        """Cast multiple DataFrame columns to Currency struct type.
        Args:
            df: Input Polars DataFrame
            columns: List of column names to convert
            currency_code: Currency code to use for conversion
        Returns:
            Polars DataFrame with specified columns converted to Currency struct
        """
        for column in columns:
            df = CurrencyConverter.cast_to_currency_struct(df, column, currency_code, new_column_name=column)
        return df
    
    @staticmethod
    def replace_column_with_currency(
        df: pl.DataFrame,
        column: str,
        currency_code: str = "USD",
        new_column_name: Optional[str] = None
    ) -> pl.DataFrame:
        """Replace a DataFrame column with a Currency struct type.
        Args:
            df: Input Polars DataFrame
            column: Column name to replace
            currency_code: Currency code to use for conversion
            new_column_name: Optional new column name for the converted data
        Returns:
            Polars DataFrame with the specified column replaced by Currency struct
        """
        return CurrencyConverter.cast_to_currency_struct(df, column, currency_code, new_column_name)
    



# Example usage
if __name__ == "__main__":
    # Test Currency class
    print("=== Testing Currency Class ===")
    usd1 = Currency(19.99, "USD")
    usd2 = Currency(10.01, "USD")
    eur1 = Currency(100.50, "EUR")
    
    print(f"USD1: {usd1}")
    print(f"USD2: {usd2}")
    print(f"EUR1: {eur1}")
    
    # Test arithmetic operations
    print(f"USD1 + USD2 = {usd1 + usd2}")
    print(f"USD1 * 2 = {usd1 * 2}")
    print(f"USD1 / 2 = {usd1 / 2}")
    
    # Test with Polars
    print("\n=== Testing with Polars ===")
    
    # Create sample data
    sample_data = [
        {"amount": 19.99, "code": "USD"},
        {"amount": 100.50, "code": "EUR"},
        {"amount": 0.01, "code": "USD"},
        {"amount": 1500.75, "code": "SCR"}
    ]
    
    # Create DataFrame
    df = create_currency_dataframe(sample_data)
    print("Original DataFrame:")
    print(df)
    
    # Perform operations
    df_with_ops = perform_currency_operations(df)
    print("\nDataFrame with operations:")
    print(df_with_ops)
    
    # Convert back to Currency objects
    currencies = CurrencyExtension.to_currency_objects(df, "price")
    print(f"\nConverted back to Currency objects:")
    for curr in currencies:
        print(f"  {curr}")
    
    # Group by currency and sum
    print("\nGrouped by currency:")
    grouped = df_with_ops.group_by("currency_code").agg([
        pl.col("amount").sum().alias("total_amount"),
        pl.col("amount_cents").sum().alias("total_cents")
    ])
    print(grouped)

=== Testing Currency Class ===
USD1: $19.9900
USD2: $10.0100
EUR1: €100.5000
USD1 + USD2 = $30.0000
USD1 * 2 = $39.9800
USD1 / 2 = $9.9950

=== Testing with Polars ===
Original DataFrame:
shape: (4, 1)
┌───────────────────────┐
│ price                 │
│ ---                   │
│ struct[3]             │
╞═══════════════════════╡
│ {19.99,"USD","$"}     │
│ {100.5,"EUR","€"}     │
│ {0.01,"USD","$"}      │
│ {1500.75,"SCR","SCR"} │
└───────────────────────┘

DataFrame with operations:
shape: (4, 6)
┌───────────────────────┬─────────┬───────────────┬────────┬─────────────────┬──────────────┐
│ price                 ┆ amount  ┆ currency_code ┆ symbol ┆ formatted_price ┆ amount_cents │
│ ---                   ┆ ---     ┆ ---           ┆ ---    ┆ ---             ┆ ---          │
│ struct[3]             ┆ f64     ┆ str           ┆ str    ┆ str             ┆ i64          │
╞═══════════════════════╪═════════╪═══════════════╪════════╪═════════════════╪══════════════╡
│ {19.99,"USD","$"}     ┆ 

In [32]:
from dataframe.emr import washing

In [50]:
df = washing.collect()

print("Original Dataframe")
print(df)
print("\nCasting 'price' column to Currency struct")
# print(f"Price column before casting:{}")

print(f"Price column dtype: {df['price'].dtype}")


print("=== Method 2: Replace price column ===")
df_replaced = CurrencyConverter.replace_column_with_currency(df, "price", "USD")
print(df_replaced)
print(f"Price column dtype: {df_replaced['price'].dtype}")
print()


df_replaced.group_by(pl.col("invoice_to")).agg(pl.col("price").sum().alias("total_price"))


Original Dataframe
shape: (2_708, 5)
┌────────────┬──────────────────┬────────────┬─────────────────────────────┬───────┐
│ date       ┆ container_number ┆ invoice_to ┆ service_remarks             ┆ price │
│ ---        ┆ ---              ┆ ---        ┆ ---                         ┆ ---   │
│ date       ┆ enum             ┆ enum       ┆ str                         ┆ i64   │
╞════════════╪══════════════════╪════════════╪═════════════════════════════╪═══════╡
│ 2025-01-04 ┆ MCAU6001185      ┆ MAERSKLINE ┆ Clean                       ┆ 30    │
│ 2025-01-04 ┆ MMAU1018360      ┆ MAERSKLINE ┆ Clean                       ┆ 30    │
│ 2025-01-04 ┆ MMAU1265502      ┆ MAERSKLINE ┆ Clean                       ┆ 30    │
│ 2025-01-04 ┆ MMAU1278644      ┆ MAERSKLINE ┆ Clean                       ┆ 30    │
│ 2025-01-04 ┆ MMAU1384076      ┆ MAERSKLINE ┆ Clean                       ┆ 30    │
│ …          ┆ …                ┆ …          ┆ …                           ┆ …     │
│ 2025-07-24 ┆ MNBU3081904  

invoice_to,total_price
enum,struct[3]
"""JMARR""",null
"""IOT""",null
"""PEVASA""",null
"""INVALID""",null
"""RAWANQ""",null
…,…
"""SAPMER""",null
"""ATUNSA""",null
"""INPESCA""",null


In [51]:
import polars as pl
from decimal import Decimal, ROUND_HALF_UP
from typing import Union, Optional, Final
import json

class Currency:
    """Currency class for handling monetary values with decimal precision."""
    
    SUPPORTED_CURRENCIES: Final[dict[str, str]] = {
        "USD": "$",
        "EUR": "€", 
        "SCR": "SCR"
    }
    
    def __init__(self, amount: Union[float, int, str, Decimal], currency_code: str = "USD"):
        # Validate currency code first
        if not isinstance(currency_code, str):
            raise ValueError("currency_code must be a string")
        if not currency_code.isupper():
            raise ValueError("currency_code must be uppercase")
        if len(currency_code) != 3:
            raise ValueError("currency_code must be exactly 3 characters long")
        if currency_code not in self.SUPPORTED_CURRENCIES:
            raise ValueError(f"currency_code must be one of the supported currencies: {list(self.SUPPORTED_CURRENCIES.keys())}")
        
        self.currency_code = currency_code
        
        # Handle amount conversion to Decimal for precision
        if not isinstance(amount, (float, int, str, Decimal)):
            raise ValueError("amount must be a float, int, string, or Decimal")
        
        # Convert to Decimal for precision, handling floats carefully
        if isinstance(amount, float):
            # Convert float to string to avoid floating point precision issues
            self.amount = Decimal(str(amount)).quantize(Decimal('0.0001'), rounding=ROUND_HALF_UP)
        else:
            self.amount = Decimal(str(amount)).quantize(Decimal('0.0001'), rounding=ROUND_HALF_UP)
    
    @property
    def symbol(self) -> str:
        return self.SUPPORTED_CURRENCIES[self.currency_code]
    
    def __str__(self) -> str:
        # Fixed formatting: "$ 30.00" instead of "$30.0000"
        return f"{self.symbol} {self.amount:.2f}"
    
    def __repr__(self) -> str:
        return f"Currency({self.amount}, '{self.currency_code}')"
    
    def __add__(self, other: 'Currency') -> 'Currency':
        if not isinstance(other, Currency):
            raise TypeError("Can only add Currency to Currency")
        if self.currency_code != other.currency_code:
            raise ValueError("Cannot add different currencies")
        return Currency(self.amount + other.amount, self.currency_code)
    
    def __radd__(self, other):
        # This enables sum() to work with Currency objects
        if other == 0:
            return self
        return self.__add__(other)
    
    def __sub__(self, other: 'Currency') -> 'Currency':
        if not isinstance(other, Currency):
            raise TypeError("Can only subtract Currency from Currency")
        if self.currency_code != other.currency_code:
            raise ValueError("Cannot subtract different currencies")
        return Currency(self.amount - other.amount, self.currency_code)
    
    def __mul__(self, other: Union[int, float, Decimal]) -> 'Currency':
        if not isinstance(other, (int, float, Decimal)):
            raise TypeError("Can only multiply Currency by numeric types")
        return Currency(self.amount * Decimal(str(other)), self.currency_code)
    
    def __truediv__(self, other: Union[int, float, Decimal]) -> 'Currency':
        if not isinstance(other, (int, float, Decimal)):
            raise TypeError("Can only divide Currency by numeric types")
        if other == 0:
            raise ZeroDivisionError("Cannot divide by zero")
        return Currency(self.amount / Decimal(str(other)), self.currency_code)
    
    def __eq__(self, other) -> bool:
        return (isinstance(other, Currency) and 
                self.currency_code == other.currency_code and 
                self.amount == other.amount)
    
    def __lt__(self, other: 'Currency') -> bool:
        if not isinstance(other, Currency):
            raise TypeError("Can only compare Currency to Currency")
        if self.currency_code != other.currency_code:
            raise ValueError("Cannot compare different currencies")
        return self.amount < other.amount
    
    def __le__(self, other: 'Currency') -> bool:
        return self == other or self < other
    
    def __gt__(self, other: 'Currency') -> bool:
        return not self <= other
    
    def __ge__(self, other: 'Currency') -> bool:
        return not self < other
    
    def to_dict(self) -> dict:
        """Convert Currency to dictionary for Polars struct compatibility."""
        return {
            "amount": float(self.amount),
            "code": self.currency_code,
            "symbol": self.symbol
        }
    
    @classmethod
    def from_dict(cls, data: dict) -> 'Currency':
        """Create Currency from dictionary."""
        return cls(data["amount"], data["code"])


# Enhanced Polars extension with aggregation support
class CurrencyPolarsOps:
    """Enhanced operations for Currency in Polars with proper aggregation support."""
    
    @staticmethod
    def sum_currency_column(df: pl.DataFrame, column: str, group_by_col: Optional[str] = None) -> pl.DataFrame:
        """
        Sum currency amounts properly, handling different currencies.
        
        Args:
            df: DataFrame with currency struct column
            column: Name of the currency column to sum
            group_by_col: Optional column to group by
            
        Returns:
            DataFrame with summed currency amounts
        """
        if group_by_col:
            return df.group_by([
                group_by_col,
                pl.col(column).struct.field("code").alias("currency_code")
            ]).agg([
                pl.col(column).struct.field("amount").sum().alias("total_amount"),
                pl.col(column).struct.field("symbol").first().alias("symbol")
            ]).with_columns([
                # Create formatted display column
                (pl.col("symbol") + " " + 
                 pl.col("total_amount").round(2).cast(pl.Utf8)).alias("formatted_total")
            ])
        else:
            return df.group_by(
                pl.col(column).struct.field("code").alias("currency_code")
            ).agg([
                pl.col(column).struct.field("amount").sum().alias("total_amount"),
                pl.col(column).struct.field("symbol").first().alias("symbol")
            ]).with_columns([
                # Create formatted display column
                (pl.col("symbol") + " " + 
                 pl.col("total_amount").round(2).cast(pl.Utf8)).alias("formatted_total")
            ])
    
    @staticmethod
    def create_currency_display_column(df: pl.DataFrame, column: str, display_col_name: str = "formatted_price") -> pl.DataFrame:
        """
        Add a formatted display column showing "$ 30.00" format.
        
        Args:
            df: DataFrame with currency struct column
            column: Name of the currency column
            display_col_name: Name for the new display column
            
        Returns:
            DataFrame with added formatted display column
        """
        return df.with_columns([
            (pl.col(column).struct.field("symbol") + " " + 
             pl.col(column).struct.field("amount").round(2).cast(pl.Utf8)).alias(display_col_name)
        ])
    
    @staticmethod
    def aggregate_currency_by_group(df: pl.DataFrame, currency_col: str, group_col: str) -> pl.DataFrame:
        """
        Aggregate currency amounts by a grouping column, properly handling multiple currencies.
        
        Args:
            df: DataFrame with currency struct column
            currency_col: Name of the currency column
            group_col: Name of the column to group by
            
        Returns:
            DataFrame with aggregated results
        """
        return (df
                .group_by([
                    pl.col(group_col),
                    pl.col(currency_col).struct.field("code").alias("currency_code")
                ])
                .agg([
                    pl.col(currency_col).struct.field("amount").sum().alias("total_amount"),
                    pl.col(currency_col).struct.field("symbol").first().alias("symbol"),
                    pl.col(currency_col).count().alias("transaction_count")
                ])
                .with_columns([
                    # Create the properly formatted display
                    (pl.col("symbol") + " " + 
                     pl.col("total_amount").round(2).cast(pl.Utf8)).alias("total_formatted")
                ])
                .sort([group_col, "currency_code"])
        )


class CurrencyConverter:
    """Utility for converting dataframe columns to currency datatype"""

    @staticmethod
    def cast_to_currency_struct(
        df: pl.DataFrame,
        column: str,
        currency_code: str = "USD",
        new_column_name: Optional[str] = None
    ) -> pl.DataFrame:
        """Cast a DataFrame column to a Currency struct type."""
        if new_column_name is None:
            new_column_name = column

        # Get currency symbol
        currency_symbol = Currency.SUPPORTED_CURRENCIES.get(currency_code)
        
        # Convert the specified column to Currency struct
        return df.with_columns(
            pl.struct([
                pl.col(column).cast(pl.Float64).alias("amount"),
                pl.lit(currency_code).alias("code"),
                pl.lit(currency_symbol).alias("symbol")
            ]).alias(new_column_name)
        )
    
    @staticmethod
    def cast_multiple_columns_to_currency(
        df: pl.DataFrame,
        columns: list[str],
        currency_code: str = "USD"
    ) -> pl.DataFrame:
        """Cast multiple DataFrame columns to Currency struct type."""
        for column in columns:
            df = CurrencyConverter.cast_to_currency_struct(df, column, currency_code, new_column_name=column)
        return df


# Example usage demonstrating the solutions
if __name__ == "__main__":
    print("=== Testing Enhanced Currency with Polars ===")
    
    # Create sample data with mixed currencies
    sample_data = pl.DataFrame({
        "invoice_to": ["Company A", "Company B", "Company A", "Company C", "Company B"],
        "price_amount": [19.99, 100.50, 25.75, 1500.00, 45.25],
        "currency": ["USD", "EUR", "USD", "SCR", "EUR"]
    })
    
    print("Original DataFrame:")
    print(sample_data)
    
    # Convert price column to currency struct
    df_with_currency = sample_data.with_columns([
        pl.struct([
            pl.col("price_amount").alias("amount"),
            pl.col("currency").alias("code"),
            pl.when(pl.col("currency") == "USD").then(pl.lit("$"))
            .when(pl.col("currency") == "EUR").then(pl.lit("€"))
            .when(pl.col("currency") == "SCR").then(pl.lit("SCR"))
            .otherwise(pl.lit("$")).alias("symbol")
        ]).alias("price")
    ]).drop(["price_amount", "currency"])
    
    print("\nDataFrame with currency struct:")
    print(df_with_currency)
    
    # Add formatted display column
    df_with_display = CurrencyPolarsOps.create_currency_display_column(
        df_with_currency, "price", "formatted_price"
    )
    
    print("\nDataFrame with formatted display:")
    print(df_with_display.select(["invoice_to", "formatted_price"]))
    
    # Now the aggregation works properly!
    print("\n=== SOLUTION: Proper Currency Aggregation ===")
    result = CurrencyPolarsOps.aggregate_currency_by_group(
        df_with_currency, "price", "invoice_to"
    )
    
    print("Aggregated results by invoice_to:")
    print(result.select(["invoice_to", "currency_code", "total_formatted", "transaction_count"]))
    
    # Alternative: Simple sum by extracting amounts
    print("\n=== Alternative: Extract amounts for simple sum ===")
    simple_sum = (df_with_currency
                  .with_columns([
                      pl.col("price").struct.field("amount").alias("amount"),
                      pl.col("price").struct.field("code").alias("currency_code")
                  ])
                  .group_by(["invoice_to", "currency_code"])
                  .agg(pl.col("amount").sum().alias("total_amount"))
    )
    print(simple_sum)
    
    # Test with Currency objects directly
    print("\n=== Testing Currency objects directly ===")
    currencies = [
        Currency(19.99, "USD"),
        Currency(100.50, "EUR"), 
        Currency(25.75, "USD")
    ]
    
    for curr in currencies:
        print(f"Currency display: {curr}")  # Shows "$ 19.99" format
    
    # Sum Currency objects directly (now works with __radd__)
    usd_currencies = [c for c in currencies if c.currency_code == "USD"]
    total_usd = sum(usd_currencies, Currency(0, "USD"))
    print(f"Total USD: {total_usd}")  # Shows "$ 45.74"

=== Testing Enhanced Currency with Polars ===
Original DataFrame:
shape: (5, 3)
┌────────────┬──────────────┬──────────┐
│ invoice_to ┆ price_amount ┆ currency │
│ ---        ┆ ---          ┆ ---      │
│ str        ┆ f64          ┆ str      │
╞════════════╪══════════════╪══════════╡
│ Company A  ┆ 19.99        ┆ USD      │
│ Company B  ┆ 100.5        ┆ EUR      │
│ Company A  ┆ 25.75        ┆ USD      │
│ Company C  ┆ 1500.0       ┆ SCR      │
│ Company B  ┆ 45.25        ┆ EUR      │
└────────────┴──────────────┴──────────┘

DataFrame with currency struct:
shape: (5, 2)
┌────────────┬──────────────────────┐
│ invoice_to ┆ price                │
│ ---        ┆ ---                  │
│ str        ┆ struct[3]            │
╞════════════╪══════════════════════╡
│ Company A  ┆ {19.99,"USD","$"}    │
│ Company B  ┆ {100.5,"EUR","€"}    │
│ Company A  ┆ {25.75,"USD","$"}    │
│ Company C  ┆ {1500.0,"SCR","SCR"} │
│ Company B  ┆ {45.25,"EUR","€"}    │
└────────────┴──────────────────────┘

Dat